# Benchmark Model 1 — ResNet-18 (General Baseline)
**EmpathBot · CS731 Group 14 | Preeti · TJ · Kanishka**

---

### What this notebook does
Fine-tunes a **pre-trained ResNet-18** on the EmpathBot 6-class emotion dataset.  
ResNet-18 is a *general-purpose* image classifier — it was **not** designed for facial expression recognition (FER).  
We train it as a baseline so we can prove that our purpose-built EmpathBot model does better.

### What we expect
- ResNet-18 will likely hit **50–65% accuracy** on the test set
- Our main EmpathBot model (trained later) should beat this
- If ResNet-18 somehow beats our model, that means our architecture choices need revisiting

### Inputs
| File | What it is |
|------|------------|
| `data/master_split.csv` | One row per image — has `filepath`, `label`, `split`, `source` columns |
| `data/class_weights.json` | Pre-computed inverse-frequency weights for CrossEntropyLoss |
| `data/processed/affectnet/` | Processed & cropped AffectNet images |
| `data/processed/rafdb/` | Processed & cropped RAF-DB images |

### Outputs
| File | What it is |
|------|------------|
| `models/resnet18_best.pt` | Best checkpoint (highest val accuracy) |
| `results/resnet18_results.json` | Final test accuracy, F1, per-class metrics |
| `results/resnet18_confusion_matrix.png` | Confusion matrix plot |
| `results/resnet18_training_curves.png` | Loss & accuracy over epochs |

---
## Cell 1 — Imports and output folders

In [1]:
import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)
from tqdm.notebook import tqdm

# ── output directories ──────────────────────────────────────────────────────
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

# ── device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

if DEVICE.type == "cpu":
    print("⚠️  Running on CPU — training will be slow (~2–4 min per epoch).")
    print("   Tip: use Google Colab with a T4 GPU to finish in minutes.")
else:
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.11.0 | Device: cpu
⚠️  Running on CPU — training will be slow (~2–4 min per epoch).
   Tip: use Google Colab with a T4 GPU to finish in minutes.


---
## Cell 2 — Config (change these if you want to experiment)

In [2]:
# ── paths ────────────────────────────────────────────────────────────────────
MASTER_CSV      = Path("data/master_split.csv")
CLASS_WEIGHTS   = Path("data/class_weights.json")
CHECKPOINT_PATH = Path("models/resnet18_best.pt")
RESULTS_PATH    = Path("results/resnet18_results.json")

# ── training hyperparameters ─────────────────────────────────────────────────
NUM_CLASSES   = 6          # neutral, trust_relief, sadness, fear_anxiety, confusion, distrust
EPOCHS        = 40         # 30–50 as specified in the task list
BATCH_SIZE    = 32         # reduce to 16 if you get out-of-memory on CPU
IMG_SIZE      = 224        # ResNet-18 expects 224×224
LR            = 1e-4       # learning rate for fine-tuning (low = don't destroy pre-trained features)
LR_DECAY_STEP = 10         # reduce LR every N epochs
LR_DECAY_GAMMA= 0.5        # multiply LR by this at each step
NUM_WORKERS   = 0          # set to 4 if on Linux/Mac with enough RAM; keep 0 on Windows
SEED          = 42

# ── EmpathBot class labels ────────────────────────────────────────────────────
CLASS_NAMES = {
    0: "neutral",
    1: "trust_relief",
    2: "sadness",
    3: "fear_anxiety",
    4: "confusion",
    5: "distrust",
}

# ── reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Config loaded:")
print(f"  Epochs       : {EPOCHS}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Image size   : {IMG_SIZE}×{IMG_SIZE}")
print(f"  Learning rate: {LR}")
print(f"  Classes      : {list(CLASS_NAMES.values())}")

Config loaded:
  Epochs       : 40
  Batch size   : 32
  Image size   : 224×224
  Learning rate: 0.0001
  Classes      : ['neutral', 'trust_relief', 'sadness', 'fear_anxiety', 'confusion', 'distrust']


---
## Cell 3 — Load master_split.csv and inspect the data

In [3]:
df = pd.read_csv(MASTER_CSV)

print(f"master_split.csv loaded — {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
print()

# -- show the split breakdown
print("Rows per split:")
print(df["split"].value_counts().to_string())
print()

# -- show class distribution in training set
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

print(f"train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}")
print()

print("Class distribution (train split):")
for label_id, name in CLASS_NAMES.items():
    count = (train_df["eb_label"] == label_id).sum()
    bar   = "█" * (count // 300)
    print(f"  {label_id} {name:<15}: {count:>5}  {bar}")

# -- sanity check: confirm path column exists and a sample file is reachable
sample = train_df.iloc[0]
sample_path = Path(sample["path"])
print()
print(f"Sample filepath: {sample_path}")
print(f"File exists    : {sample_path.exists()}")
if not sample_path.exists():
    print("⚠️  File not found! Check that the path column matches your actual folder structure.")

df.head(3)

master_split.csv loaded — 36,749 rows
Columns: ['path', 'dataset', 'orig_label', 'eb_label', 'split', 'face_ratio']

Rows per split:
split
train    26997
val       5483
test      4073
eval       196

train: 26,997 | val: 5,483 | test: 4,073

Class distribution (train split):
  0 neutral        :  7868  ██████████████████████████
  1 trust_relief   :  4935  ████████████████
  2 sadness        :  2913  █████████
  3 fear_anxiety   :  2539  ████████
  4 confusion      :  3329  ███████████
  5 distrust       :  5413  ██████████████████

Sample filepath: data/processed/affectnet/0/015563.jpg
File exists    : True


,path,dataset,orig_label,eb_label,split,face_ratio
0,data/processed/affectnet/0/015563.jpg,affectnet,0,0,train,0.399719
1,data/processed/affectnet/0/019359.jpg,affectnet,0,0,train,0.313614
2,data/processed/affectnet/0/018047.jpg,affectnet,0,0,train,0.363281


---
## Cell 4 — Load class weights for imbalanced training

In [4]:
with open(CLASS_WEIGHTS) as f:
    weights_dict = json.load(f)

# class_weights.json stores class names as keys — convert to ordered tensor
# Expected format: {"neutral": 0.49, "trust_relief": 0.78, ...}
# We rebuild as a list ordered by class ID (0–5)
name_to_id = {v: k for k, v in CLASS_NAMES.items()}

weight_list = []
print("Class weights for CrossEntropyLoss:")
print(f"  {'ID':<4} {'Class':<20} {'Weight'}")
print("  " + "-"*35)
for label_id in range(NUM_CLASSES):
    name   = CLASS_NAMES[label_id]
    weight = weights_dict.get(str(label_id), weights_dict.get(name, 1.0))
    weight_list.append(float(weight))
    print(f"  {label_id:<4} {name:<20} {weight:.4f}")

CLASS_WEIGHT_TENSOR = torch.tensor(weight_list, dtype=torch.float32).to(DEVICE)
print()
print(f"Tensor: {CLASS_WEIGHT_TENSOR}")
print("✅ Class weights ready")

Class weights for CrossEntropyLoss:
  ID   Class                Weight
  -----------------------------------
  0    neutral              0.4914
  1    trust_relief         0.7834
  2    sadness              1.3271
  3    fear_anxiety         1.5226
  4    confusion            1.1613
  5    distrust             0.7142

Tensor: tensor([0.4914, 0.7834, 1.3271, 1.5226, 1.1613, 0.7142])
✅ Class weights ready


---
## Cell 5 — Dataset class and data transforms

**Why separate train vs. val/test transforms?**
- Training: we add random flips, crops, colour jitter to make the model more robust
- Val/test: we only normalise — no randomness, so results are reproducible

In [7]:
# ── transforms ───────────────────────────────────────────────────────────────
# ImageNet mean/std — used because ResNet-18 was pre-trained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),   # slightly larger, then crop
    transforms.RandomCrop(IMG_SIZE),                     # random crop to 224
    transforms.RandomHorizontalFlip(p=0.5),              # faces can be mirrored
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
    ),                                                   # simulate different lighting
    transforms.RandomRotation(degrees=10),               # slight angle variation
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── dataset class ─────────────────────────────────────────────────────────────
class EmpathBotDataset(Dataset):
    """
    Reads image paths and labels from master_split.csv.
    Skips missing files silently and logs a warning.
    """
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        # Only keep rows where the file actually exists on disk
        valid_mask   = dataframe["path"].apply(lambda p: Path(p).exists())
        missing_count = (~valid_mask).sum()
        if missing_count > 0:
            print(f"  ⚠️  {missing_count} files not found on disk — skipping them")

        self.df        = dataframe[valid_mask].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row["path"]).convert("RGB")
        label = int(row["eb_label"])
        if self.transform:
            img = self.transform(img)
        return img, label


# ── build datasets ────────────────────────────────────────────────────────────
print("Building datasets from master_split.csv...")
train_dataset = EmpathBotDataset(train_df, transform=train_transform)
val_dataset   = EmpathBotDataset(val_df,   transform=val_transform)
test_dataset  = EmpathBotDataset(test_df,  transform=val_transform)

# ── build dataloaders ─────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

print(f"  train : {len(train_dataset):>6,} images  |  {len(train_loader):>4} batches")
print(f"  val   : {len(val_dataset):>6,} images  |  {len(val_loader):>4} batches")
print(f"  test  : {len(test_dataset):>6,} images  |  {len(test_loader):>4} batches")

# ── quick visual check: show a batch of training images ──────────────────────
imgs, labels = next(iter(train_loader))
print(f"\nBatch shape : {imgs.shape}  (batch × channels × H × W)")
print(f"Label sample: {labels[:8].tolist()}")
print(f"Label names : {[CLASS_NAMES[l.item()] for l in labels[:8]]}")

Building datasets from master_split.csv...
  train : 26,997 images  |   844 batches
  val   :  5,483 images  |   172 batches
  test  :  4,073 images  |   128 batches

Batch shape : torch.Size([32, 3, 224, 224])  (batch × channels × H × W)
Label sample: [2, 0, 5, 0, 0, 5, 5, 0]
Label names : ['sadness', 'neutral', 'distrust', 'neutral', 'neutral', 'distrust', 'distrust', 'neutral']


---
## Cell 6 — Build the model

We take a **pre-trained ResNet-18** (trained on ImageNet's 1,000 classes) and replace its final layer with a new layer that outputs 6 classes (our EmpathBot emotions).

This is called **transfer learning** — the model already knows how to detect edges, textures, and shapes from ImageNet. We just teach it to use those features for emotion recognition.

In [8]:
def build_resnet18(num_classes: int = 6, freeze_backbone: bool = False) -> nn.Module:
    """
    Load pre-trained ResNet-18 and swap the final FC layer for our 6-class head.

    Args:
        num_classes   : number of emotion classes (6)
        freeze_backbone: if True, only train the final layer (faster but worse)
                         if False, fine-tune the whole network (slower but better)
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        # Freeze all layers — only the new head will update
        for param in model.parameters():
            param.requires_grad = False

    # Replace final fully-connected layer
    # Original: Linear(512, 1000)  →  New: Linear(512, 6)
    in_features       = model.fc.in_features  # 512 for ResNet-18
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),           # dropout helps prevent overfitting
        nn.Linear(in_features, num_classes)
    )

    return model


model = build_resnet18(num_classes=NUM_CLASSES, freeze_backbone=False)
model = model.to(DEVICE)

# ── count trainable parameters ─────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("ResNet-18 loaded:")
print(f"  Total parameters    : {total_params:>10,}")
print(f"  Trainable parameters: {trainable_params:>10,}")
print(f"  Final layer         : {model.fc}")
print(f"  Device              : {DEVICE}")
print()
print("Why freeze_backbone=False?")
print("  We fine-tune the whole network so facial-specific features can develop.")
print("  Freezing only makes sense if your dataset is tiny (<1,000 images).")

ResNet-18 loaded:
  Total parameters    : 11,179,590
  Trainable parameters: 11,179,590
  Final layer         : Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=512, out_features=6, bias=True)
)
  Device              : cpu

Why freeze_backbone=False?
  We fine-tune the whole network so facial-specific features can develop.
  Freezing only makes sense if your dataset is tiny (<1,000 images).


---
## Cell 7 — Loss function, optimiser, and learning rate scheduler

In [9]:
# ── loss function ─────────────────────────────────────────────────────────────
# Weighted CrossEntropyLoss penalises mistakes on minority classes more heavily.
# This prevents the model from just always predicting "neutral" (the biggest class).
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHT_TENSOR)

# ── optimiser ─────────────────────────────────────────────────────────────────
# Adam is a safe default for fine-tuning.
# weight_decay=1e-4 adds L2 regularisation to prevent overfitting.
optimizer = optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,
)

# ── learning rate scheduler ───────────────────────────────────────────────────
# Every LR_DECAY_STEP epochs, multiply LR by LR_DECAY_GAMMA (0.5 = halve it).
# This lets the model make big updates early, then fine adjustments later.
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=LR_DECAY_STEP,
    gamma=LR_DECAY_GAMMA,
)

print("Training setup:")
print(f"  Loss        : CrossEntropyLoss (weighted)")
print(f"  Optimiser   : Adam (lr={LR}, weight_decay=1e-4)")
print(f"  LR schedule : ×{LR_DECAY_GAMMA} every {LR_DECAY_STEP} epochs")
print()
print("LR schedule preview:")
lr = LR
for ep in range(1, EPOCHS + 1):
    if ep % LR_DECAY_STEP == 0:
        lr *= LR_DECAY_GAMMA
        print(f"  Epoch {ep:>3}: LR drops to {lr:.2e}")

Training setup:
  Loss        : CrossEntropyLoss (weighted)
  Optimiser   : Adam (lr=0.0001, weight_decay=1e-4)
  LR schedule : ×0.5 every 10 epochs

LR schedule preview:
  Epoch  10: LR drops to 5.00e-05
  Epoch  20: LR drops to 2.50e-05
  Epoch  30: LR drops to 1.25e-05
  Epoch  40: LR drops to 6.25e-06


---
## Cell 8 — Training and validation loop

The loop runs one full pass through the training set, then evaluates on the validation set.  
We save the model whenever validation accuracy improves — this is called **early stopping via best-checkpoint saving**.

In [10]:
def run_epoch(model, loader, criterion, optimizer, device, is_training: bool):
    """
    Run one epoch (training or validation).
    Returns: (average_loss, accuracy)
    """
    if is_training:
        model.train()
    else:
        model.eval()

    running_loss    = 0.0
    correct         = 0
    total           = 0

    ctx = torch.enable_grad() if is_training else torch.no_grad()

    with ctx:
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)              # forward pass
            loss    = criterion(outputs, labels) # compute loss

            if is_training:
                optimizer.zero_grad()            # clear old gradients
                loss.backward()                  # compute new gradients
                optimizer.step()                 # update weights

            running_loss += loss.item() * images.size(0)
            preds         = outputs.argmax(dim=1)
            correct      += (preds == labels).sum().item()
            total        += images.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


# ── training loop ─────────────────────────────────────────────────────────────
history = {
    "train_loss": [], "train_acc": [],
    "val_loss":   [], "val_acc":   [],
    "lr":         [],
}
best_val_acc    = 0.0
best_epoch      = 0
patience_counter= 0
EARLY_STOP      = 10   # stop if val acc doesn't improve for 10 epochs in a row

print(f"Starting training — {EPOCHS} epochs on {DEVICE}")
print(f"Best checkpoint will be saved to: {CHECKPOINT_PATH}")
print("-" * 72)
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7} | {'LR':>8} | {'Status'}")
print("-" * 72)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, DEVICE, is_training=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, optimizer, DEVICE, is_training=False)

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    # save best model
    status = ""
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        best_epoch       = epoch
        patience_counter = 0
        torch.save({
            "epoch":      epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc":    val_acc,
            "val_loss":   val_loss,
            "class_names": CLASS_NAMES,
            "num_classes": NUM_CLASSES,
        }, CHECKPOINT_PATH)
        status = "✅ saved"
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP:
            print(f"\nEarly stopping triggered at epoch {epoch} (no improvement for {EARLY_STOP} epochs)")
            break

    epoch_time = time.time() - epoch_start
    print(
        f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.1%} | "
        f"{val_loss:>8.4f} | {val_acc:>6.1%} | {current_lr:>8.1e} | {status}"
    )

total_time = time.time() - start_time
print("-" * 72)
print(f"Training complete in {total_time/60:.1f} minutes")
print(f"Best val accuracy: {best_val_acc:.1%} at epoch {best_epoch}")
print(f"Checkpoint saved : {CHECKPOINT_PATH}")

Starting training — 40 epochs on cpu
Best checkpoint will be saved to: models/resnet18_best.pt
------------------------------------------------------------------------
 Epoch | Train Loss | Train Acc | Val Loss | Val Acc |       LR | Status
------------------------------------------------------------------------
     1 |     0.9272 |    66.7% |   0.6833 |  73.2% |  1.0e-04 | ✅ saved


KeyboardInterrupt: 

---
## Cell 9 — Plot training curves

If the train accuracy keeps going up but val accuracy plateaus or drops → the model is **overfitting**  
If both train and val accuracy are stuck low → the model is **underfitting** (try a higher LR or more epochs)

In [ ]:
actual_epochs = len(history["train_loss"])
epoch_range   = range(1, actual_epochs + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("ResNet-18 Baseline — Training Curves", fontsize=14, fontweight="bold")

# -- accuracy
ax = axes[0]
ax.plot(epoch_range, [a * 100 for a in history["train_acc"]], label="Train", color="steelblue")
ax.plot(epoch_range, [a * 100 for a in history["val_acc"]],   label="Val",   color="tomato",   linestyle="--")
ax.axvline(best_epoch, color="gray", linestyle=":", linewidth=1, label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%d%%"))
ax.grid(True, alpha=0.3)

# -- loss
ax = axes[1]
ax.plot(epoch_range, history["train_loss"], label="Train", color="steelblue")
ax.plot(epoch_range, history["val_loss"],   label="Val",   color="tomato",   linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# -- learning rate
ax = axes[2]
ax.plot(epoch_range, history["lr"], color="purple")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate Schedule")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/resnet18_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: results/resnet18_training_curves.png")

---
## Cell 10 — Evaluate on the test set

We load the **best checkpoint** (not the final epoch) and evaluate on the held-out test set.  
The test set was never seen during training or validation — this is the honest measure of real-world performance.

In [ ]:
# ── load best checkpoint ──────────────────────────────────────────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} (val acc: {checkpoint['val_acc']:.1%})")

# ── run inference on test set ─────────────────────────────────────────────────
model.eval()
all_preds  = []
all_labels = []
all_probs  = []    # softmax probabilities — useful for confidence analysis

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating on test set"):
        images  = images.to(DEVICE)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        preds   = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# ── overall metrics ───────────────────────────────────────────────────────────
test_accuracy = accuracy_score(all_labels, all_preds)
macro_f1      = f1_score(all_labels, all_preds, average="macro")
weighted_f1   = f1_score(all_labels, all_preds, average="weighted")

print()
print("="*50)
print(" TEST SET RESULTS — ResNet-18 Baseline")
print("="*50)
print(f"  Overall accuracy  : {test_accuracy:.1%}")
print(f"  Macro F1          : {macro_f1:.3f}")
print(f"  Weighted F1       : {weighted_f1:.3f}")
print()

# ── per-class breakdown ───────────────────────────────────────────────────────
class_name_list = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
report = classification_report(
    all_labels, all_preds,
    target_names=class_name_list,
    digits=3,
)
print("Per-class classification report:")
print(report)

---
## Cell 11 — Confusion matrix

The confusion matrix shows **which emotions the model confuses with each other**.  
The diagonal (top-left to bottom-right) = correct predictions.  
Off-diagonal = mistakes. Look at which pairs get confused most — this is useful for your report.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

# Normalise so each row sums to 1 (shows % of each class correctly/incorrectly classified)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("ResNet-18 Baseline — Confusion Matrix", fontsize=13, fontweight="bold")

for ax, data, title, fmt in [
    (axes[0], cm,      "Raw counts",  "d"),
    (axes[1], cm_norm, "Normalised",  ".2f"),
]:
    sns.heatmap(
        data,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=class_name_list,
        yticklabels=class_name_list,
        ax=ax,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title(title)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=9)

plt.tight_layout()
plt.savefig("results/resnet18_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: results/resnet18_confusion_matrix.png")

# ── highlight the worst confusions ───────────────────────────────────────────
print("\nTop 5 most common misclassifications:")
cm_copy = cm_norm.copy()
np.fill_diagonal(cm_copy, 0)   # zero the diagonal so we see only errors
flat     = cm_copy.flatten()
top_idx  = flat.argsort()[-5:][::-1]
for idx in top_idx:
    true_class = idx // NUM_CLASSES
    pred_class = idx  % NUM_CLASSES
    print(f"  Actual {CLASS_NAMES[true_class]:<15} → Predicted {CLASS_NAMES[pred_class]:<15}: {cm_norm[true_class, pred_class]:.1%}")

---
## Cell 12 — Confidence score distribution

The model outputs a softmax probability for each class. The highest one is the predicted class.  
Low confidence (e.g. below 40%) = the model is uncertain — this is when EmpathBot should ask a clarifying question instead of acting on the predicted emotion.

In [ ]:
# max confidence = probability assigned to the predicted class
max_probs = all_probs.max(axis=1)
is_correct = (all_preds == all_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("ResNet-18 — Confidence Score Distribution", fontsize=13, fontweight="bold")

# -- overall confidence histogram
ax = axes[0]
ax.hist(max_probs[is_correct],  bins=30, alpha=0.7, label="Correct",   color="steelblue")
ax.hist(max_probs[~is_correct], bins=30, alpha=0.7, label="Incorrect", color="tomato")
ax.axvline(0.4, color="black", linestyle="--", linewidth=1.2, label="40% threshold")
ax.set_xlabel("Max softmax probability")
ax.set_ylabel("Count")
ax.set_title("Confidence: correct vs incorrect predictions")
ax.legend()
ax.grid(True, alpha=0.3)

# -- per-class average confidence
ax = axes[1]
class_conf = [
    max_probs[all_labels == c].mean() if (all_labels == c).sum() > 0 else 0
    for c in range(NUM_CLASSES)
]
bars = ax.bar(class_name_list, class_conf, color="steelblue", edgecolor="white")
ax.axhline(0.4, color="black", linestyle="--", linewidth=1.2, label="40% threshold")
for bar, conf in zip(bars, class_conf):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{conf:.0%}", ha="center", fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Average max confidence")
ax.set_title("Average confidence per class")
ax.set_xticklabels(class_name_list, rotation=20, ha="right", fontsize=9)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("results/resnet18_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

low_conf_pct = (max_probs < 0.4).mean()
print(f"Predictions below 40% confidence: {low_conf_pct:.1%}")
print("→ These would trigger EmpathBot's 'ask a clarifying question' mode")

---
## Cell 13 — Save results to JSON (for the comparison table)

In [ ]:
from sklearn.metrics import precision_score, recall_score

per_class_precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
per_class_recall    = recall_score(   all_labels, all_preds, average=None, zero_division=0)
per_class_f1        = f1_score(       all_labels, all_preds, average=None, zero_division=0)

results = {
    "model":            "ResNet-18 (baseline)",
    "dataset":          "AffectNet-HQ + RAF-DB (EmpathBot 6-class mapping)",
    "test_size":        int(len(all_labels)),
    "best_epoch":       best_epoch,
    "best_val_acc":     round(best_val_acc, 4),
    "test_accuracy":    round(float(test_accuracy), 4),
    "macro_f1":         round(float(macro_f1), 4),
    "weighted_f1":      round(float(weighted_f1), 4),
    "per_class": {}
}

for i, name in CLASS_NAMES.items():
    results["per_class"][name] = {
        "precision": round(float(per_class_precision[i]), 4),
        "recall":    round(float(per_class_recall[i]),    4),
        "f1":        round(float(per_class_f1[i]),        4),
        "support":   int((all_labels == i).sum()),
    }

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to {RESULTS_PATH}")
print()
print("── Summary for comparison table ──────────────────────────────")
print(f"  Model           : {results['model']}")
print(f"  Test accuracy   : {results['test_accuracy']:.1%}")
print(f"  Macro F1        : {results['macro_f1']:.3f}")
print(f"  Weighted F1     : {results['weighted_f1']:.3f}")
print()
print(f"  {'Class':<18} {'Precision':>9} {'Recall':>9} {'F1':>6}")
print(f"  {'-'*46}")
for name, metrics in results["per_class"].items():
    print(f"  {name:<18} {metrics['precision']:>9.3f} {metrics['recall']:>9.3f} {metrics['f1']:>6.3f}")

print()
target = 0.70
if test_accuracy >= target:
    print(f"✅ Beats the 70% target — ResNet-18 is a strong baseline!")
    print(f"   This means our EmpathBot model must do even better to justify the design.")
else:
    print(f"📊 Accuracy is {test_accuracy:.1%} (below 70% target) — this is expected for a general baseline.")
    print(f"   Our purpose-built EmpathBot model should close this gap.")

---
## Cell 14 — What to write in your report

Run this cell to get a pre-filled summary paragraph for your individual report.

In [ ]:
with open(RESULTS_PATH) as f:
    r = json.load(f)

worst_class  = min(r["per_class"], key=lambda k: r["per_class"][k]["f1"])
best_class   = max(r["per_class"], key=lambda k: r["per_class"][k]["f1"])

print("─" * 70)
print("REPORT PARAGRAPH (copy-paste and rewrite in your own words)")
print("─" * 70)
print(f"""
ResNet-18 Baseline Results

As a general-purpose baseline, we fine-tuned a pre-trained ResNet-18 
(He et al., 2016) on the EmpathBot 6-class emotion dataset, which 
combines AffectNet-HQ and RAF-DB mapped to our custom label set.
ResNet-18 was chosen as a baseline because it is a standard image 
classification backbone that was not designed for facial expression 
recognition (FER).

The model achieved {r['test_accuracy']:.1%} overall accuracy on the 
held-out test set, with a macro-averaged F1 of {r['macro_f1']:.3f}. 
Per-class analysis revealed that the model performed best on 
'{best_class}' (F1 = {r['per_class'][best_class]['f1']:.3f}) and worst 
on '{worst_class}' (F1 = {r['per_class'][worst_class]['f1']:.3f}), 
suggesting that general-purpose features are insufficient for reliably 
detecting subtle or minority emotions in elderly users.

These results establish a baseline against which our purpose-built 
Emotional Affect Model will be compared, with the expectation that 
FER-specific design choices (augmentation, weighted loss, backbone 
selection) should produce meaningful improvements.
""")
print("─" * 70)
print()
print("Files produced:")
for f in ["models/resnet18_best.pt",
          "results/resnet18_results.json",
          "results/resnet18_confusion_matrix.png",
          "results/resnet18_training_curves.png",
          "results/resnet18_confidence.png"]:
    exists = "✅" if Path(f).exists() else "❌ MISSING"
    print(f"  {exists}  {f}")